In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Load your CSV
csv_path = r"C:\Users\japje\Documents\B.G\wiki_hybrid_chunks_300.csv"
df = pd.read_csv(csv_path)
chunks = df["chunk_text"].tolist()

# Load TinyLLaMA model and tokenizer (adjust device if you want to use GPU)
tinyllama_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(tinyllama_model_id)
model = AutoModelForCausalLM.from_pretrained(tinyllama_model_id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Helper function to generate question for one chunk
def generate_question(chunk_text, max_new_tokens=40):
    prompt = f"""Based on the following text, generate one factual question.

Text:
{chunk_text[:500]}

Question:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        num_return_sequences=1,
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract question after "Question:"
    question = generated_text.split("Question:")[-1].strip().split("\n")[0]
    return question

# Generate questions for first 100 chunks (or change as needed)
generated_questions = []

for i, chunk in enumerate(chunks[:100]):
    try:
        question = generate_question(chunk)
        generated_questions.append({"chunk_id": df.iloc[i]["chunk_id"], "question": question})
        print(f"[{i+1}] {question}")
    except Exception as e:
        print(f"Error on chunk {i+1}: {e}")

# Save generated questions to CSV
questions_df = pd.DataFrame(generated_questions)
questions_df.to_csv("generated_questions_tinyllama.csv2", index=False)
print("Saved questions to generated_questions_tinyllama.csv2")


[1] Who provides the voice of the animated lead characters in the TV series Delilah julius and Pearlie?
[2] How did the filmography of Marieve Herington include two more albums and two theme songs for television programs?
[3] How do you describe Honoka's character as a force of nature?
[4] Which Canadian actor is known for his roles in the movies "The Naked Gun" and "The Naked Gun 2½: The Smell of Fear"?
[5] What was the name of the minor league team with which Robert Brown shuffled after his stints with the Hartford Whalers, Chicago Blackhawks, and Dallas Stars?
[6] Who was the starting right winger for the 40th national hockey penguins league all star game in the following season?
[7] What was the name of the Edmonton Oilers pay per view, 2 and what is his current position as a hockey instructor?
[8] Which team has the most points in the NHL, based on the provided data?
[9] What is the name of the oilers hockey institute instructor mentioned in the text material and what is his role 